# Chapter 6.5: LLM as Feature Encoder

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** how LLMs can serve as feature encoders for recommendation systems
2. **Implement** KAR-style knowledge-augmented user/item profiles
3. **Build** RLMRec-inspired representation learning with language model features
4. **Generate** text-based features for CTR prediction models
5. **Compare** frozen vs fine-tuned LLM feature encoders
6. **Integrate** LLM-generated features into existing recommendation pipelines
7. **Evaluate** the impact of text-enhanced features on recommendation quality

## Prerequisites

- Understanding of CTR prediction models (Part 2)
- Familiarity with text embeddings and transformers
- Chapter 6.4 (LLM foundations for rec)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part6/chapter_6.5_llm_feature_encoder.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part6/chapter_6.5_llm_feature_encoder.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cpu')
print(f'Using device: {device}')

## 1. LLM as Feature Encoder: Overview

Instead of using LLMs to directly generate recommendations (Chapter 6.4), we can use them as **feature encoders** that produce rich representations for existing recommendation models.

### Three Paradigms

1. **KAR** (Xi et al., 2023, Alibaba): Use LLM to generate knowledge-augmented profiles, then extract features
2. **RLMRec** (Ren et al., 2024): Align LLM representations with collaborative filtering embeddings
3. **Text-enhanced CTR**: Encode item text (reviews, descriptions) as additional features

### Why LLM Features?

- **Rich semantics**: LLMs capture nuanced meaning from text
- **World knowledge**: Pre-trained LLMs bring external knowledge about items
- **Cold-start**: Text features are available even for new items
- **Cross-domain transfer**: Language representations transfer across domains

> **💡 Concept:** The key insight is that LLMs and recommendation models excel at different things. LLMs understand language and world knowledge; rec models understand user behavior. Combining them gives the best of both worlds.

In [ ]:
# Create synthetic dataset with text features
np.random.seed(42)

n_users = 2000
n_items = 500
n_user_features = 8  # age, gender, etc.
n_item_features = 12  # category, price, etc.
text_embed_dim = 64  # Simulated LLM embedding dimension

# User features
user_features = np.random.randn(n_users, n_user_features).astype(np.float32) * 0.5

# Item features (categorical + numerical)
item_features = np.random.randn(n_items, n_item_features).astype(np.float32) * 0.5

# Simulated LLM text embeddings for items
# In practice, these come from encoding item descriptions with an LLM
n_categories = 10
item_categories = np.random.randint(0, n_categories, n_items)

# Text embeddings have structure: items in same category have similar embeddings
category_centers = np.random.randn(n_categories, text_embed_dim) * 1.0
item_text_embeddings = np.zeros((n_items, text_embed_dim), dtype=np.float32)
for i in range(n_items):
    item_text_embeddings[i] = category_centers[item_categories[i]] + np.random.randn(text_embed_dim) * 0.3

# Generate click labels
user_latent = np.random.randn(n_users, 16) * 0.5
item_latent = np.random.randn(n_items, 16) * 0.5

# Generate training data (user, item, label)
n_samples = 50000
user_ids = np.random.randint(0, n_users, n_samples)
item_ids = np.random.randint(0, n_items, n_samples)

# Labels based on latent factors + text similarity
logits = np.sum(user_latent[user_ids] * item_latent[item_ids], axis=1)
# Add text-based signal
user_text_pref = np.random.randn(n_users, text_embed_dim) * 0.3
text_signal = np.sum(user_text_pref[user_ids] * item_text_embeddings[item_ids], axis=1) * 0.1
logits += text_signal
labels = (logits + np.random.randn(n_samples) * 0.5 > 0).astype(np.float32)

# Train/test split
split = int(0.8 * n_samples)
train_idx = np.arange(split)
test_idx = np.arange(split, n_samples)

print(f'Users: {n_users}, Items: {n_items}')
print(f'Training samples: {len(train_idx)}, Test samples: {len(test_idx)}')
print(f'Positive rate: {labels.mean():.4f}')
print(f'Text embedding dim: {text_embed_dim}')

## 2. Simulated LLM Feature Generation

### KAR: Knowledge-Augmented Recommendation

**KAR** (Xi et al., SIGIR 2023, Alibaba) uses LLMs to generate structured knowledge about users and items:

1. **User Profile**: "Based on this user's history of buying electronics and books, generate a preference profile."
2. **Item Profile**: "Describe this product's key attributes, target audience, and use cases."
3. **Factual Knowledge**: "What is the relationship between this brand and quality?"

These text outputs are then encoded into features.

In [ ]:
class SimulatedLLMEncoder:
    """Simulates LLM text encoding for features.
    
    In production, this would call a real LLM API or use a local model.
    Here we simulate the process to demonstrate the pipeline.
    """
    
    def __init__(self, embed_dim=64, n_categories=10):
        np.random.seed(123)
        self.embed_dim = embed_dim
        # Simulate pre-trained category knowledge
        self.category_knowledge = np.random.randn(n_categories, embed_dim) * 0.8
        # Simulate relationship knowledge
        self.cross_category = np.random.randn(n_categories, n_categories) * 0.2
    
    def encode_item_description(self, item_idx, item_category):
        """Simulate encoding an item's text description."""
        # Base category embedding + item-specific noise
        base = self.category_knowledge[item_category]
        noise = np.random.randn(self.embed_dim) * 0.15
        return base + noise
    
    def generate_user_profile_features(self, user_item_history, item_categories):
        """KAR-style: generate user profile from interaction history.
        
        Simulates: "This user frequently buys [categories]. They prefer [attributes]."
        """
        if len(user_item_history) == 0:
            return np.zeros(self.embed_dim)
        
        # Aggregate category preferences
        cat_counts = np.zeros(len(self.category_knowledge))
        for item_id in user_item_history:
            cat_counts[item_categories[item_id]] += 1
        cat_probs = cat_counts / (cat_counts.sum() + 1e-8)
        
        # Weighted combination of category knowledge
        profile = cat_probs @ self.category_knowledge
        return profile.astype(np.float32)
    
    def generate_item_knowledge(self, item_idx, item_category):
        """KAR-style: generate factual knowledge about an item.
        
        Simulates: "This [category] product is known for [attributes]. Similar to [related items]."
        """
        base = self.category_knowledge[item_category]
        # Add cross-category knowledge
        cross = self.cross_category[item_category] @ self.category_knowledge * 0.1
        return (base + cross + np.random.randn(self.embed_dim) * 0.1).astype(np.float32)


# Generate LLM features for all items
llm_encoder = SimulatedLLMEncoder(embed_dim=text_embed_dim)

item_llm_features = np.zeros((n_items, text_embed_dim), dtype=np.float32)
item_knowledge_features = np.zeros((n_items, text_embed_dim), dtype=np.float32)

for i in range(n_items):
    item_llm_features[i] = llm_encoder.encode_item_description(i, item_categories[i])
    item_knowledge_features[i] = llm_encoder.generate_item_knowledge(i, item_categories[i])

# Generate user profile features
# First, build user histories
user_item_histories = defaultdict(list)
for idx in train_idx:
    if labels[idx] > 0:
        user_item_histories[user_ids[idx]].append(item_ids[idx])

user_llm_features = np.zeros((n_users, text_embed_dim), dtype=np.float32)
for u in range(n_users):
    user_llm_features[u] = llm_encoder.generate_user_profile_features(
        user_item_histories[u], item_categories
    )

print(f'Item LLM features shape: {item_llm_features.shape}')
print(f'Item knowledge features shape: {item_knowledge_features.shape}')
print(f'User LLM profile features shape: {user_llm_features.shape}')

## 3. CTR Model: Baseline (Without LLM Features)

In [ ]:
class CTRModel(nn.Module):
    """Simple DeepFM-style CTR model."""
    
    def __init__(self, n_users, n_items, user_feat_dim, item_feat_dim,
                 embed_dim=16, hidden_dims=(128, 64)):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.item_embed = nn.Embedding(n_items, embed_dim)
        
        total_dim = embed_dim * 2 + user_feat_dim + item_feat_dim
        
        layers = []
        prev_dim = total_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, user_ids, item_ids, user_feats, item_feats):
        u_emb = self.user_embed(user_ids)
        i_emb = self.item_embed(item_ids)
        x = torch.cat([u_emb, i_emb, user_feats, item_feats], dim=1)
        return self.mlp(x).squeeze(-1)


def train_and_evaluate(model, train_data, test_data, n_epochs=20, lr=1e-3):
    """Train CTR model and return AUC."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    train_loader = DataLoader(TensorDataset(*train_data), batch_size=512, shuffle=True)
    
    history = {'train_loss': [], 'test_auc': []}
    
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            u_ids, i_ids, u_feat, i_feat, y = batch
            logits = model(u_ids, i_ids, u_feat, i_feat)
            loss = F.binary_cross_entropy_with_logits(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                u_ids, i_ids, u_feat, i_feat, y = test_data
                logits = model(u_ids, i_ids, u_feat, i_feat)
                probs = torch.sigmoid(logits).cpu().numpy()
                y_np = y.cpu().numpy()
                # Simple AUC approximation
                from sklearn.metrics import roc_auc_score, log_loss
                auc = roc_auc_score(y_np, probs)
                logloss = log_loss(y_np, np.clip(probs, 1e-7, 1-1e-7))
            
            history['train_loss'].append(total_loss / len(train_loader))
            history['test_auc'].append(auc)
            print(f'Epoch {epoch+1:3d} | Loss: {total_loss/len(train_loader):.4f} | '
                  f'AUC: {auc:.4f} | LogLoss: {logloss:.4f}')
    
    return history


# Prepare baseline data (no LLM features)
def prepare_data(user_ids, item_ids, labels, user_features, item_features, indices):
    u_ids = torch.LongTensor(user_ids[indices])
    i_ids = torch.LongTensor(item_ids[indices])
    u_feat = torch.FloatTensor(user_features[user_ids[indices]])
    i_feat = torch.FloatTensor(item_features[item_ids[indices]])
    y = torch.FloatTensor(labels[indices])
    return u_ids, i_ids, u_feat, i_feat, y


train_data_base = prepare_data(user_ids, item_ids, labels, user_features, item_features, train_idx)
test_data_base = prepare_data(user_ids, item_ids, labels, user_features, item_features, test_idx)

print('=== Baseline CTR Model (No LLM Features) ===')
baseline_model = CTRModel(n_users, n_items, n_user_features, n_item_features)
baseline_history = train_and_evaluate(baseline_model, train_data_base, test_data_base)

## 4. CTR Model with LLM Features

Now let's add LLM-generated features as additional inputs to the CTR model.

> **🔑 Pro Tip:** In practice, LLM features are pre-computed offline and cached. The inference-time cost is just a lookup, not an LLM forward pass.

In [ ]:
class CTRModelWithLLM(nn.Module):
    """CTR model enhanced with LLM-generated features."""
    
    def __init__(self, n_users, n_items, user_feat_dim, item_feat_dim,
                 llm_user_dim, llm_item_dim, embed_dim=16, hidden_dims=(128, 64),
                 llm_proj_dim=32):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, embed_dim)
        self.item_embed = nn.Embedding(n_items, embed_dim)
        
        # LLM feature projections (dimensionality reduction)
        self.user_llm_proj = nn.Sequential(
            nn.Linear(llm_user_dim, llm_proj_dim),
            nn.ReLU(),
            nn.LayerNorm(llm_proj_dim)
        )
        self.item_llm_proj = nn.Sequential(
            nn.Linear(llm_item_dim, llm_proj_dim),
            nn.ReLU(),
            nn.LayerNorm(llm_proj_dim)
        )
        
        total_dim = embed_dim * 2 + user_feat_dim + item_feat_dim + llm_proj_dim * 2
        
        layers = []
        prev_dim = total_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.mlp = nn.Sequential(*layers)
    
    def forward(self, user_ids, item_ids, user_feats, item_feats,
                user_llm_feats, item_llm_feats):
        u_emb = self.user_embed(user_ids)
        i_emb = self.item_embed(item_ids)
        u_llm = self.user_llm_proj(user_llm_feats)
        i_llm = self.item_llm_proj(item_llm_feats)
        x = torch.cat([u_emb, i_emb, user_feats, item_feats, u_llm, i_llm], dim=1)
        return self.mlp(x).squeeze(-1)


def prepare_data_with_llm(user_ids, item_ids, labels, user_features, item_features,
                           user_llm_features, item_llm_features, indices):
    u_ids = torch.LongTensor(user_ids[indices])
    i_ids = torch.LongTensor(item_ids[indices])
    u_feat = torch.FloatTensor(user_features[user_ids[indices]])
    i_feat = torch.FloatTensor(item_features[item_ids[indices]])
    u_llm = torch.FloatTensor(user_llm_features[user_ids[indices]])
    i_llm = torch.FloatTensor(item_llm_features[item_ids[indices]])
    y = torch.FloatTensor(labels[indices])
    return u_ids, i_ids, u_feat, i_feat, u_llm, i_llm, y


def train_and_evaluate_llm(model, train_data, test_data, n_epochs=20, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    train_loader = DataLoader(TensorDataset(*train_data), batch_size=512, shuffle=True)
    history = {'train_loss': [], 'test_auc': []}
    
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            u_ids, i_ids, u_feat, i_feat, u_llm, i_llm, y = batch
            logits = model(u_ids, i_ids, u_feat, i_feat, u_llm, i_llm)
            loss = F.binary_cross_entropy_with_logits(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                u_ids, i_ids, u_feat, i_feat, u_llm, i_llm, y = test_data
                logits = model(u_ids, i_ids, u_feat, i_feat, u_llm, i_llm)
                probs = torch.sigmoid(logits).cpu().numpy()
                y_np = y.cpu().numpy()
                from sklearn.metrics import roc_auc_score
                auc = roc_auc_score(y_np, probs)
            history['train_loss'].append(total_loss / len(train_loader))
            history['test_auc'].append(auc)
            print(f'Epoch {epoch+1:3d} | Loss: {total_loss/len(train_loader):.4f} | AUC: {auc:.4f}')
    
    return history


# Prepare data with LLM features
train_data_llm = prepare_data_with_llm(
    user_ids, item_ids, labels, user_features, item_features,
    user_llm_features, item_llm_features, train_idx
)
test_data_llm = prepare_data_with_llm(
    user_ids, item_ids, labels, user_features, item_features,
    user_llm_features, item_llm_features, test_idx
)

print('\n=== CTR Model WITH LLM Features ===')
llm_model = CTRModelWithLLM(
    n_users, n_items, n_user_features, n_item_features,
    llm_user_dim=text_embed_dim, llm_item_dim=text_embed_dim
)
llm_history = train_and_evaluate_llm(llm_model, train_data_llm, test_data_llm)

In [ ]:
# Compare baseline vs LLM-enhanced
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

eval_epochs = list(range(5, 21, 5))

axes[0].plot(eval_epochs, baseline_history['test_auc'], 'b-o', label='Baseline CTR', markersize=6)
axes[0].plot(eval_epochs, llm_history['test_auc'], 'r-s', label='CTR + LLM Features', markersize=6)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUC')
axes[0].set_title('AUC: Baseline vs LLM-Enhanced CTR')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(eval_epochs, baseline_history['train_loss'], 'b-o', label='Baseline', markersize=6)
axes[1].plot(eval_epochs, llm_history['train_loss'], 'r-s', label='+ LLM Features', markersize=6)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Training Loss')
axes[1].set_title('Training Loss Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nFinal AUC - Baseline: {baseline_history["test_auc"][-1]:.4f}')
print(f'Final AUC - LLM Enhanced: {llm_history["test_auc"][-1]:.4f}')
print(f'Improvement: {(llm_history["test_auc"][-1] - baseline_history["test_auc"][-1]):.4f}')

## 5. Frozen vs Fine-tuned LLM Encoder

A critical design choice: should the LLM encoder be **frozen** or **fine-tuned**?

| Approach | Pros | Cons |
|----------|------|------|
| **Frozen** | Fast, no extra GPU memory, preserves world knowledge | May not align with rec task |
| **Fine-tuned** | Better task alignment, can learn rec-specific patterns | Expensive, may lose general knowledge |
| **Adapter** | Balance: fine-tune small adapters only | Middle ground in cost and quality |

> **⚠️ Common Pitfall:** Fine-tuning a large LLM on sparse recommendation data can lead to catastrophic forgetting — the model loses its pre-trained world knowledge. Use adapter-based approaches or very small learning rates.

In [ ]:
class SimulatedLLMWithAdapter(nn.Module):
    """Simulates frozen LLM + trainable adapter for feature encoding."""
    
    def __init__(self, input_dim, embed_dim=64, adapter_dim=16):
        super().__init__()
        # Frozen LLM layers (simulated)
        self.frozen_layer1 = nn.Linear(input_dim, embed_dim)
        self.frozen_layer2 = nn.Linear(embed_dim, embed_dim)
        
        # Freeze
        for p in self.frozen_layer1.parameters():
            p.requires_grad = False
        for p in self.frozen_layer2.parameters():
            p.requires_grad = False
        
        # Trainable adapter (bottleneck)
        self.adapter = nn.Sequential(
            nn.Linear(embed_dim, adapter_dim),
            nn.ReLU(),
            nn.Linear(adapter_dim, embed_dim)
        )
        # Initialize adapter to near-identity
        nn.init.zeros_(self.adapter[2].weight)
        nn.init.zeros_(self.adapter[2].bias)
    
    def forward(self, x):
        # Frozen forward
        h = F.relu(self.frozen_layer1(x))
        h = F.relu(self.frozen_layer2(h))
        # Adapter (residual)
        h = h + self.adapter(h)
        return h


# Compare frozen vs adapter approaches
approaches = {
    'Frozen (no adapter)': False,
    'With Adapter': True
}

results = {}
for name, use_adapter in approaches.items():
    torch.manual_seed(42)
    
    if use_adapter:
        encoder = SimulatedLLMWithAdapter(n_item_features, text_embed_dim, adapter_dim=16)
    else:
        encoder = SimulatedLLMWithAdapter(n_item_features, text_embed_dim, adapter_dim=16)
        # Disable adapter training
        for p in encoder.adapter.parameters():
            p.requires_grad = False
    
    trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
    total = sum(p.numel() for p in encoder.parameters())
    print(f'{name}: {trainable:,} trainable / {total:,} total params')
    results[name] = {'trainable': trainable, 'total': total}

# Visualize parameter comparison
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
names = list(results.keys())
trainable = [results[n]['trainable'] for n in names]
frozen = [results[n]['total'] - results[n]['trainable'] for n in names]

ax.barh(names, frozen, label='Frozen', color='lightblue')
ax.barh(names, trainable, left=frozen, label='Trainable', color='coral')
ax.set_xlabel('Number of Parameters')
ax.set_title('Frozen vs Adapter: Parameter Comparison')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 6. RLMRec: Alignment Learning

**RLMRec** (Ren et al., 2024) aligns LLM representations with collaborative filtering embeddings using contrastive learning:

$$\mathcal{L}_{\text{align}} = -\log \frac{\exp(\text{sim}(\mathbf{h}_i^{\text{LLM}}, \mathbf{h}_i^{\text{CF}}) / \tau)}{\sum_j \exp(\text{sim}(\mathbf{h}_i^{\text{LLM}}, \mathbf{h}_j^{\text{CF}}) / \tau)}$$

This ensures the LLM features are in the same semantic space as the CF embeddings.

In [ ]:
class AlignmentModule(nn.Module):
    """RLMRec-style alignment between LLM and CF representations."""
    
    def __init__(self, llm_dim, cf_dim, proj_dim=32, temperature=0.1):
        super().__init__()
        self.llm_proj = nn.Sequential(
            nn.Linear(llm_dim, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )
        self.cf_proj = nn.Sequential(
            nn.Linear(cf_dim, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )
        self.temperature = temperature
    
    def contrastive_loss(self, llm_features, cf_features):
        """InfoNCE contrastive loss for alignment."""
        z_llm = F.normalize(self.llm_proj(llm_features), dim=1)
        z_cf = F.normalize(self.cf_proj(cf_features), dim=1)
        
        # Similarity matrix
        sim = z_llm @ z_cf.T / self.temperature
        
        # Labels: diagonal entries are positive pairs
        labels = torch.arange(sim.size(0), device=sim.device)
        
        # Symmetric loss
        loss = (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2
        return loss


# Train alignment
cf_embed_dim = 16
# Simulate CF embeddings from a trained model
cf_embeddings = np.random.randn(n_items, cf_embed_dim).astype(np.float32)
# Make CF embeddings somewhat correlated with categories
for i in range(n_items):
    cf_embeddings[i] += np.random.randn(cf_embed_dim) * 0.1
    cf_embeddings[i, item_categories[i] % cf_embed_dim] += 0.5

align_module = AlignmentModule(text_embed_dim, cf_embed_dim, proj_dim=32)
align_optimizer = torch.optim.Adam(align_module.parameters(), lr=1e-3)

llm_feats_t = torch.FloatTensor(item_llm_features)
cf_feats_t = torch.FloatTensor(cf_embeddings)

align_losses = []
for epoch in range(100):
    # Sample batch
    idx = np.random.choice(n_items, 128, replace=False)
    batch_llm = llm_feats_t[idx]
    batch_cf = cf_feats_t[idx]
    
    loss = align_module.contrastive_loss(batch_llm, batch_cf)
    align_optimizer.zero_grad()
    loss.backward()
    align_optimizer.step()
    align_losses.append(loss.item())
    
    if (epoch + 1) % 25 == 0:
        print(f'Alignment Epoch {epoch+1} | Loss: {loss.item():.4f}')

plt.figure(figsize=(8, 4))
plt.plot(align_losses, alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Contrastive Loss')
plt.title('RLMRec-style Alignment Training')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

### 🏋️ Exercise 1: Implement Feature Gating for LLM Features

Not all LLM features are equally useful. Implement a gating mechanism to automatically weight them.

In [ ]:
# TODO: Implement gated fusion of LLM and traditional features
class GatedFeatureFusion(nn.Module):
    """
    TODO: Implement a gating mechanism:
    1. Compute gate = sigmoid(W * [traditional_feat, llm_feat] + b)
    2. fused = gate * llm_proj + (1 - gate) * traditional_proj
    3. Use fused features in CTR model
    """
    def __init__(self, trad_dim, llm_dim, output_dim):
        super().__init__()
        # TODO: Implement
        pass
    
    def forward(self, trad_feat, llm_feat):
        # TODO: Implement
        pass

# TODO: Train CTR model with gated fusion and compare against simple concatenation

### 🏋️ Exercise 2: Cold-Start Analysis

Show that LLM features are especially valuable for cold-start items.

In [ ]:
# TODO: Cold-start experiment
# 1. Split items into "warm" (>50 interactions) and "cold" (<5 interactions)
# 2. Train baseline CTR model on warm items only
# 3. Train LLM-enhanced CTR model on warm items only
# 4. Evaluate both on cold items
# 5. Show that LLM features help more for cold items
# 6. Visualize the AUC gap (warm vs cold) for both models

### 🏋️ Exercise 3: Multi-Modal LLM Features

Combine text features from different sources (title, description, reviews).

In [ ]:
# TODO: Multi-source feature fusion
# 1. Simulate three types of LLM features:
#    - Title embeddings (shorter, more structured)
#    - Description embeddings (richer, more diverse)
#    - Review embeddings (user-generated, noisy)
# 2. Implement attention-based fusion across the three sources
# 3. Compare: single source vs attention-fused vs concatenated
# 4. Analyze: which source contributes most via attention weights?

## Summary

| Approach | Method | Key Benefit | Year |
|----------|--------|------------|------|
| **KAR** | LLM-generated knowledge profiles | Rich factual features | 2023 |
| **RLMRec** | Contrastive alignment | Unified representation space | 2024 |
| **Text-enhanced CTR** | LLM text embeddings as features | Cold-start handling | 2023+ |
| **Adapter-based** | Frozen LLM + small adapters | Efficient fine-tuning | 2023+ |

**Key Takeaways:**
1. LLM features complement (not replace) traditional recommendation features
2. Pre-computing LLM features offline makes deployment practical
3. Frozen LLMs with small adapters offer the best cost-quality trade-off
4. LLM features are especially valuable for cold-start items/users
5. Alignment between LLM and CF spaces is critical for feature quality